In [ ]:
import os
from dotenv import load_dotenv
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import requests
import pandas as pd
import time

# Load keys safely from your .env file
load_dotenv()

# --- SPOTIFY SETUP ---
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=os.getenv("SPOTIFY_CLIENT_ID"), 
    client_secret=os.getenv("SPOTIFY_CLIENT_SECRET")
))

# --- RAPIDAPI (MUSICAE) SETUP ---
RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")
RAPIDAPI_HOST = "spotify-extended-audio-features-api.p.rapidapi.com"

headers = {
    "x-rapidapi-key": RAPIDAPI_KEY,
    "x-rapidapi-host": RAPIDAPI_HOST
}

# --- LOAD KWORB DATA ---

final_dataset = []

for index, row in kworb_df.iterrows():
    song_title = row['Song']
    artist_name = row['Artist']
    
    query = f"{song_title} {artist_name}"
    
    #Get the Spotify Track ID
    search_result = sp.search(q=query, type='track', limit=1)
    
    if search_result['tracks']['items']:
        track = search_result['tracks']['items'][0]
        track_id = track['id']
        
        # Pass ID to Musicae via RapidAPI
        url = f"https://{RAPIDAPI_HOST}/v1/audio-features/{track_id}"
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            audio_features = response.json()
            
            final_dataset.append({
                'Kworb Rank / Streams': row.get('Streams'), # Keep whatever metric Kworb gave you
                'Track': track['name'],
                'Artist': track['artists'][0]['name'],
                'Release Date': track['album']['release_date'],
                'Danceability': audio_features.get('danceability'),
                'Energy': audio_features.get('energy'),
                'Valence': audio_features.get('valence'),
                'Tempo': audio_features.get('tempo'),
                'Key': audio_features.get('key')
            })
            
    #Avoids rate limits
    time.sleep(0.5) 

# Convert everything into your final master DataFrame
master_df = pd.DataFrame(final_dataset)
print(master_df.head())